# Assignment 7: Signal Processing and Spectral Analysis
Please submit this assignment to Canvas as a jupyter notebook (.ipynb). This assignment uses data from the Delayed Free Recall of Word Lists study (https://openneuro.org/datasets/ds004789/versions/3.1.0).

## 📥 Saving your answers for grading

This assignment is **auto-graded**. After each question there is a **grader cell**
that saves the data you plotted or computed into an `answers/Module_15/` folder so
it can be compared against the reference answers.

**For each question:**
1. The question tells you which result(s) to produce and the expected data
   structure/format. Do your analysis and bind each result to a variable.
2. In the grader cell, replace the placeholder variable with **your** variable name.
3. Run the grader cell — it calls `save_answer(...)` and writes your answer.

Your **plots are saved too** — make sure each plotting cell calls `plt.show()` so the
figure can be captured for the grade report.

Make sure every grader cell runs without error before you submit. You don't need to
change anything else.


In [ ]:
# grader setup — enables saving the figures you plot (run once, early)
try:
    from grader.answer_io import enable_figure_capture
    enable_figure_capture()
except Exception as _e:
    print("grader: figure capture not enabled:", _e)


In [2]:
# imports
import mne
import ptsa
import sys
sys.path.insert(0, 'dependencies/bidsreader')
from bidsreader import CMLBIDSReader as BIDSReader
import numpy as np
import matplotlib.pyplot as plt
from ptsa.data.filters import morlet
from ptsa.data.filters import ButterworthFilter

## Assignment Overview

Much like the previous assignment, in this project you will analyze multi-session free recall experiments to determine the spectral biomarkers of successful memory encoding.  Again, to determine differences in brain activity (EEG signals) that predict whether a studied item will be subsequently remembered (recalled) you will compare two classes of events: word encodings that led to later recall and those that did not.  This assignment will ask you to analyze data reported in Solomon et al (2018), a study was carried out in patients with temporal lobe epilepsy undergoing a neurosurgical evaluation that required implantation of depth electrodes through small openings in the skull.

Notably, in this assignment, you will analyze intracranial EEG data.  A key step in the processing of these data involves using CT and MRI scans to localize the electrodes and then matching up these localizations to a standard brain map based on the average of many patients’ brains to assign each electrode a brain region. Fortunately for you, this step has already been taken care of by the researchers who assembled these data. All you need to know is how to obtain the location tags for each electrode in each patient. 

You will analyze data from patients in 'intrac_subs'. Each of these patients was chosen because they have at least three sessions of Fr1/CatFR1 data and their recall performance was at least 30%. For each of these subjects, use the following processing steps:
* Load EEG with mne-bids XXXfrom a bipolar montage loaded using XXX STILL NEED TO FIGURE OUT
* Apply a Butterworth notch filter around 60 Hz (freqs = [58, 62]) when extracting the voltage.
* Calculate power at the above frequencies with a Morlet wavelet with wavenumber (keyword “width”) of 6 for each encoding event (from time 0 until 1.6 seconds after the encoding event onset) using a 1 second buffer.
* For each frequency, channel, and encoding event, average the power over the entire 1600 ms encoding period (but not over the buffer period!)
* Log-transform the average encoding power values (using base-10 logarithms, e.g. np.log10).
* Compute the within-session z-transform of these power values as in the previous problems.
* In some cases you may notice artifacts in the data that manifest in power values of zero. These would produce problems in the transformation and classification, so please exclude any events with this issue from all analyses.

In [ ]:
# use these subjects for Part II
intrac_subs = ['R1060M', 'R1061T', 'R1065J', 'R1066P', 'R1092J', 
               'R1108J', 'R1111M', 'R1207J', 'R1226D', 'R1230J', 
               'R1236J', 'R1243T', 'R1247P', 'R1292E', 'R1308T',
               'R1310J', 'R1328E', 'R1332M', 'R1334T', 'R1337E', 
               'R1341T', 'R1354E', 'R1395M', 'R1486J', 'R1495J', 
               'R1501J', 'R1542J']

freqs = np.unique(np.round(np.logspace(np.log10(1), np.log10(300), 17)))

## Question 1
Find all subjects (from 'intrac_subs') who have electrodes localized to the following three regions: temporal lobe, frontal lobe, and hippocampus. The ind.region columns with labels ’bankssts’ and ’temporal’ in them are in the temporal lobe while labels 'frontal' and 'hippocampus' correspond to frontal lobe and hippocampus respectively.

1) Create a table with the number of electrodes in each of these regions for each subject.

<!-- grader-note -->
> **📥 For grading, produce and save:** `region_counts` (dataframe: subject, temporal, frontal, hippo — table of average electrode (bipolar) counts per subject in temporal, frontal, hippocampal regions) → Q1.1. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

In [ ]:
# lobe labels for temporal/frontal/hippocampal regions
locations = [['temporal', 'bankssts'], 'frontal', 'hippocampal']

In [ ]:
# Question 1.1
### YOUR CODE HERE

In [ ]:
# ── grader cell (Question 1.1) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q1.1_region_counts", region_counts, module=15, question="1.1")   # ← replace `region_counts` with your variable

## Question 2
Assess the subsequent memory effect at a particular frequency across locations in the brain. 
* Instead of topographic maps, just analyze the data for channels pooled within the frontal and temporal brain regions.
* Conduct these analyses on only those subjects who have at least one electrode in each of the temporal lobe and frontal lobe regions.
    * Plot these two regions in the resulting figure.
* Because subjects will have different numbers of electrodes in each of these regions, you need to decide how to combine the data across electrodes.
* Complete the analysis for both 147 Hz and 10 Hz.

1) Think of a reasonable way to combine data across electrodes and try it.  Make plots to display the SME.
2) Think of a another reasonable way to combine data across electrodes and try it.  Make plots to display the SME.
3) Report the approaches that you used, as well as the strengths and weaknesses of each approach.  There may not be a "right" answer here ... use your best judgement and argue for the method you determine is best.

<!-- grader-note -->
> **📥 For grading, produce and save:** `rec_means_10hz` (array (2,) — recalled mean power z-scores [temporal, frontal] at 10 Hz, bipolar reference (errorbar y)) → Q2.2; `rec_sems_10hz` (array (2,) — recalled SEM [temporal, frontal] at 10 Hz, bipolar reference (errorbar yerr)) → Q2.2; `nrec_means_10hz` (array (2,) — not-recalled mean power z-scores [temporal, frontal] at 10 Hz, bipolar reference) → Q2.2; `nrec_sems_10hz` (array (2,) — not-recalled SEM [temporal, frontal] at 10 Hz, bipolar reference) → Q2.2; `rec_means_147hz` (array (2,) — recalled mean power z-scores [temporal, frontal] at 147 Hz, bipolar reference) → Q2.2; `rec_sems_147hz` (array (2,) — recalled SEM [temporal, frontal] at 147 Hz, bipolar reference) → Q2.2; `nrec_means_147hz` (array (2,) — not-recalled mean power z-scores [temporal, frontal] at 147 Hz, bipolar reference) → Q2.2; `nrec_sems_147hz` (array (2,) — not-recalled SEM [temporal, frontal] at 147 Hz, bipolar reference) → Q2.2. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

In [2]:
# Question 2.1
### YOUR CODE HERE

In [3]:
# Question 2.2
### YOUR CODE HERE

In [ ]:
# ── grader cell (Question 2.2) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q2.2_rec_means_10hz", rec_means_10hz, module=15, question="2.2", fig="last")   # ← replace `rec_means_10hz` with your variable
save_answer("Q2.2_rec_sems_10hz", rec_sems_10hz, module=15, question="2.2")   # ← replace `rec_sems_10hz` with your variable
save_answer("Q2.2_nrec_means_10hz", nrec_means_10hz, module=15, question="2.2")   # ← replace `nrec_means_10hz` with your variable
save_answer("Q2.2_nrec_sems_10hz", nrec_sems_10hz, module=15, question="2.2")   # ← replace `nrec_sems_10hz` with your variable
save_answer("Q2.2_rec_means_147hz", rec_means_147hz, module=15, question="2.2")   # ← replace `rec_means_147hz` with your variable
save_answer("Q2.2_rec_sems_147hz", rec_sems_147hz, module=15, question="2.2")   # ← replace `rec_sems_147hz` with your variable
save_answer("Q2.2_nrec_means_147hz", nrec_means_147hz, module=15, question="2.2")   # ← replace `nrec_means_147hz` with your variable
save_answer("Q2.2_nrec_sems_147hz", nrec_sems_147hz, module=15, question="2.2")   # ← replace `nrec_sems_147hz` with your variable

## Question 3
For the preceding analysis, you used bipolar referencing. As with scalp EEG you can use either bipolar or an average reference. With intracranial EEG, however, the average reference is not always well defined because each subject will have their own idiosyncratic arrangement of electrodes, with some regions, such as medial temporal lobe and lateral temporal cortex, being systematically selected for the clinical treatment of epilepsy. For this problem:
 * Create an average reference by first computing an average across electrodes within each lobe of the brain being analyzed, then average the averages across the lobes but only for lobes that have at least 5 electrodes. 
    * This will ensure that each lobe is equally weighted within the overall average.

1) Recompute the analysis in the previous problem using this method (again separately at 10 Hz and 147 Hz). 
2) Compare your results and comment on the strengths and weaknesses of each.

<!-- grader-note -->
> **📥 For grading, produce and save:** `rec_means_10hz` (array (2,) — recalled mean power z-scores [temporal, frontal] at 10 Hz, region-weighted average reference) → Q3.1; `rec_sems_10hz` (array (2,) — recalled SEM [temporal, frontal] at 10 Hz, average reference) → Q3.1; `nrec_means_10hz` (array (2,) — not-recalled mean power z-scores [temporal, frontal] at 10 Hz, average reference) → Q3.1; `nrec_sems_10hz` (array (2,) — not-recalled SEM [temporal, frontal] at 10 Hz, average reference) → Q3.1; `rec_means_147hz` (array (2,) — recalled mean power z-scores [temporal, frontal] at 147 Hz, average reference) → Q3.1; `rec_sems_147hz` (array (2,) — recalled SEM [temporal, frontal] at 147 Hz, average reference) → Q3.1; `nrec_means_147hz` (array (2,) — not-recalled mean power z-scores [temporal, frontal] at 147 Hz, average reference) → Q3.1; `nrec_sems_147hz` (array (2,) — not-recalled SEM [temporal, frontal] at 147 Hz, average reference) → Q3.1. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

In [4]:
# Question 3.1
### YOUR CODE HERE

In [ ]:
# ── grader cell (Question 3.1) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q3.1_rec_means_10hz", rec_means_10hz, module=15, question="3.1", fig="last")   # ← replace `rec_means_10hz` with your variable
save_answer("Q3.1_rec_sems_10hz", rec_sems_10hz, module=15, question="3.1")   # ← replace `rec_sems_10hz` with your variable
save_answer("Q3.1_nrec_means_10hz", nrec_means_10hz, module=15, question="3.1")   # ← replace `nrec_means_10hz` with your variable
save_answer("Q3.1_nrec_sems_10hz", nrec_sems_10hz, module=15, question="3.1")   # ← replace `nrec_sems_10hz` with your variable
save_answer("Q3.1_rec_means_147hz", rec_means_147hz, module=15, question="3.1")   # ← replace `rec_means_147hz` with your variable
save_answer("Q3.1_rec_sems_147hz", rec_sems_147hz, module=15, question="3.1")   # ← replace `rec_sems_147hz` with your variable
save_answer("Q3.1_nrec_means_147hz", nrec_means_147hz, module=15, question="3.1")   # ← replace `nrec_means_147hz` with your variable
save_answer("Q3.1_nrec_sems_147hz", nrec_sems_147hz, module=15, question="3.1")   # ← replace `nrec_sems_147hz` with your variable

Question 3.2

**YOUR ANSWER HERE**